In [ ]:
import os
import heapq
import random
import numpy as np
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

GRID_SIZE = 64
NUM_SAMPLES = 5000

# 障碍物密度
OBSTACLE_PROB = 0.25

# start 和 goal 至少相隔这么远（Manhattan distance）
MIN_START_GOAL_DISTANCE = 30

OUTPUT_DIR = "dataset"

SEED = 42


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# A* Search
# ============================================================

def heuristic(a, b):
    """Manhattan distance."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(obstacle_map, start, goal):
    """
    obstacle_map:
        [H, W]
        1 = obstacle
        0 = free

    start, goal:
        (row, col)

    return:
        list of coordinates representing shortest path,
        or None if no path exists.
    """

    H, W = obstacle_map.shape

    open_heap = []

    # heap element:
    # (f_score, g_score, current_node)
    heapq.heappush(
        open_heap,
        (heuristic(start, goal), 0, start)
    )

    came_from = {}

    g_score = {
        start: 0
    }

    visited = set()

    directions = [
        (-1, 0),   # up
        (1, 0),    # down
        (0, -1),   # left
        (0, 1)     # right
    ]

    while open_heap:

        f, g, current = heapq.heappop(open_heap)

        if current in visited:
            continue

        visited.add(current)

        # Found goal
        if current == goal:

            path = [current]

            while current in came_from:
                current = came_from[current]
                path.append(current)

            path.reverse()

            return path

        r, c = current

        for dr, dc in directions:

            nr = r + dr
            nc = c + dc

            # out of boundary
            if nr < 0 or nr >= H or nc < 0 or nc >= W:
                continue

            # obstacle
            if obstacle_map[nr, nc] == 1:
                continue

            neighbor = (nr, nc)

            tentative_g = g_score[current] + 1

            if (
                neighbor not in g_score
                or tentative_g < g_score[neighbor]
            ):

                came_from[neighbor] = current

                g_score[neighbor] = tentative_g

                f_score = (
                    tentative_g
                    + heuristic(neighbor, goal)
                )

                heapq.heappush(
                    open_heap,
                    (
                        f_score,
                        tentative_g,
                        neighbor
                    )
                )

    return None


# ============================================================
# Random Map Generation
# ============================================================

def generate_random_obstacle_map(
    size=64,
    obstacle_prob=0.25
):
    """
    Random binary obstacle map.

    return:
        [H, W]
        1 = obstacle
        0 = free
    """

    obstacle_map = (
        np.random.rand(size, size)
        < obstacle_prob
    ).astype(np.float32)

    return obstacle_map


# ============================================================
# Start / Goal Sampling
# ============================================================

def sample_start_goal(
    obstacle_map,
    min_distance=30
):
    """
    Randomly choose start and goal from free cells.
    """

    free_cells = np.argwhere(
        obstacle_map == 0
    )

    if len(free_cells) < 2:
        return None, None

    for _ in range(100):

        indices = np.random.choice(
            len(free_cells),
            size=2,
            replace=False
        )

        start = tuple(
            free_cells[indices[0]]
        )

        goal = tuple(
            free_cells[indices[1]]
        )

        distance = heuristic(
            start,
            goal
        )

        if distance >= min_distance:
            return start, goal

    return None, None


# ============================================================
# Create one map-path pair
# ============================================================

def generate_sample():

    while True:

        # --------------------------------
        # 1. Generate obstacle map
        # --------------------------------

        obstacle_map = (
            generate_random_obstacle_map(
                size=GRID_SIZE,
                obstacle_prob=OBSTACLE_PROB
            )
        )

        # --------------------------------
        # 2. Sample start + goal
        # --------------------------------

        start, goal = sample_start_goal(
            obstacle_map,
            min_distance=MIN_START_GOAL_DISTANCE
        )

        if start is None:
            continue

        # explicitly guarantee
        # start / goal are free
        obstacle_map[start] = 0
        obstacle_map[goal] = 0

        # --------------------------------
        # 3. Run A*
        # --------------------------------

        path = astar(
            obstacle_map,
            start,
            goal
        )

        # no valid path
        if path is None:
            continue

        # Optional:
        # prevent extremely trivial paths
        if len(path) < MIN_START_GOAL_DISTANCE:
            continue

        # --------------------------------
        # 4. Build 3-channel map
        # --------------------------------

        map_tensor = np.zeros(
            (
                3,
                GRID_SIZE,
                GRID_SIZE
            ),
            dtype=np.float32
        )

        # obstacle channel
        map_tensor[0] = obstacle_map

        # start channel
        map_tensor[
            1,
            start[0],
            start[1]
        ] = 1.0

        # goal channel
        map_tensor[
            2,
            goal[0],
            goal[1]
        ] = 1.0

        # --------------------------------
        # 5. Build path mask
        # --------------------------------

        path_tensor = np.zeros(
            (
                1,
                GRID_SIZE,
                GRID_SIZE
            ),
            dtype=np.float32
        )

        for r, c in path:

            path_tensor[
                0,
                r,
                c
            ] = 1.0

        return (
            map_tensor,
            path_tensor,
            start,
            goal,
            len(path)
        )


# ============================================================
# Generate Dataset
# ============================================================

def generate_dataset():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    print(
        f"Generating {NUM_SAMPLES} samples..."
    )

    for i in tqdm(
        range(NUM_SAMPLES)
    ):

        (
            map_tensor,
            path_tensor,
            start,
            goal,
            path_length
        ) = generate_sample()

        filename = os.path.join(
            OUTPUT_DIR,
            f"pair_{i:05d}.npz"
        )

        np.savez_compressed(
            filename,

            map=map_tensor,

            path=path_tensor,

            # metadata
            start=np.array(
                start,
                dtype=np.int64
            ),

            goal=np.array(
                goal,
                dtype=np.int64
            ),

            path_length=np.array(
                path_length,
                dtype=np.int64
            )
        )

    print(
        f"\nDone."
    )

    print(
        f"Dataset saved to: {OUTPUT_DIR}"
    )


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    generate_dataset()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Load one sample
# ============================================================

sample_path = "dataset/pair_00000.npz"

data = np.load(sample_path)

grid_map = data["map"]       # [3, 64, 64]
path = data["path"]          # [1, 64, 64]

start = data["start"]
goal = data["goal"]
path_length = data["path_length"]


print("Map shape:", grid_map.shape)
print("Path shape:", path.shape)
print("Start:", start)
print("Goal:", goal)
print("A* path length:", path_length)


# ============================================================
# Extract channels
# ============================================================

obstacle = grid_map[0]
start_map = grid_map[1]
goal_map = grid_map[2]

path_map = path[0]


# ============================================================
# Visualize
# ============================================================

plt.figure(figsize=(8, 8))

# 1 = obstacle → black
# 0 = free → white
plt.imshow(
    obstacle,
    cmap="gray_r",
    origin="upper"
)

# Get path coordinates
path_y, path_x = np.where(path_map == 1)

plt.scatter(
    path_x,
    path_y,
    s=10,
    label="A* Path"
)

# Start
plt.scatter(
    start[1],
    start[0],
    s=100,
    marker="o",
    label="Start"
)

# Goal
plt.scatter(
    goal[1],
    goal[0],
    s=120,
    marker="*",
    label="Goal"
)

plt.title(
    f"Map + A* Ground Truth Path\n"
    f"Path length = {int(path_length)}"
)

plt.legend()
plt.axis("off")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("dataset/pair_00000.npz")

grid_map = data["map"]
path = data["path"]

obstacle = grid_map[0]
start_map = grid_map[1]
goal_map = grid_map[2]
path_map = path[0]


fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(obstacle, cmap="gray_r")
axes[0].set_title("Obstacle")

axes[1].imshow(start_map, cmap="gray_r")
axes[1].set_title("Start")

axes[2].imshow(goal_map, cmap="gray_r")
axes[2].set_title("Goal")

axes[3].imshow(path_map, cmap="gray_r")
axes[3].set_title("A* Ground Truth Path")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()